# **Урок 1**. Изображение и классические методы работы с ним

## **План**
1. Как работать с файлами в opencv
2. Поработаем с HSV моделью
3. Выравниваем гистограммы
4. Разбираемся с Resize
5. Выделяем границы и контуры
6. Гомография

Наверняка у вас уже есть numpy в окружении. Сейчас для работы нам понадобится ещё библиотека opencv. 

Устанавливается она следующим образом:
```
pip install opencv-python
```

А вот дока: https://pypi.org/project/opencv-python/

## **Как работать с файлами в opencv**

In [ ]:
# импортируем немножко библиотек, которые нам понадобятся

import cv2 # это opencv, не удивляйтесь
import numpy as np
import matplotlib

from matplotlib import pyplot as plt
matplotlib.rcParams['figure.figsize'] = (10, 8) # установим дефолтный размер для всех картинок, чтобы каждый раз не писать

In [ ]:
# чекнем версию либы
cv2.__version__

In [ ]:
# функция чтения изображения
img = cv2.imread('naruto.png')

In [ ]:
# и что там у нас внутри?
img

In [ ]:
# смотрим на размеры изображения
h, w, c = img.shape
print(f'h={h}, w={w}, c={c}')

In [ ]:
# посмотрим, что за красоту мы считали
plt.axis('off')
plt.imshow(img)
plt.show()

In [ ]:
# В opencv по умолчанию изображения считываются в BGR, поэтому надо провести конвертацию один из способов:

# дока -> https://docs.opencv.org/4.x/df/d9d/tutorial_py_colorspaces.html
img_rgb_opencv = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# либо просто в обратном порядке пересохраним каналы по axis=2
img_rgb = img[:, :, ::-1]

In [ ]:
# проверяем что получилось одно и то же
fig, ax = plt.subplots(ncols=2, figsize=(15, 20))
ax[0].axis('off')
ax[0].imshow(img_rgb_opencv)
ax[0].title.set_text('cv2.cvtColor(img, cv2.COLOR_BGR2RGB)')
ax[1].axis('off')
ax[1].imshow(img_rgb)
ax[1].title.set_text('img[:, :, ::-1]')
plt.show()

Теперь давайте посмотрим, как с помощью opencv изображение можно сохранить.  

In [ ]:
# сделаем кроп нашего изображения
img_crop = img_rgb[500:800, 500:800]
plt.imshow(img_crop)
plt.axis('off')
plt.show()

In [ ]:
# а теперь сохраним его
# важно не забыть тут вернуть каналы в BGR порядок (!)
cv2.imwrite('naruto_crop.png', img_crop[:, :, ::-1])

Давайте посмотрим отдельно как выглядят R,G,B каналы в нашем изображении. 

In [ ]:
# Визуализируем отдельно R канал (он 0й)
fig, ax = plt.subplots(ncols=2, figsize=(15, 20))
ax[0].axis('off')
ax[0].imshow(img_rgb[:, :, 0])
ax[0].title.set_text("cmap='viridis'")
ax[1].axis('off')
ax[1].imshow(img_rgb[:, :, 0], cmap='gray')
ax[1].title.set_text("cmap='gray'")
plt.show()

In [ ]:
# отображение так происходит, так как тензор для pyplot одноканальный
# чтобы увидеть интенсивность R,G,B цветов, тензор должен стать снова трехканальным
# для этого давайте по очереди занулим все каналы, кроме одного желаемого 

fig, ax = plt.subplots(ncols=3, figsize=(20, 25))

for C in [0, 1, 2]:
    temp = np.zeros(img.shape, dtype='uint8')
    temp[:, :, C] = img[:, :, C]
    ax[C].axis('off')
    ax[C].imshow(temp)

plt.show()

In [ ]:
# мы уже говорили, что grayscale представление получается не просто усреднением
# формулу можно найти в доке: https://docs.opencv.org/3.4/de/d25/imgproc_color_conversions.html
# для всех конвертаций есть две версии: BGR2... и RGB2...

fig, ax = plt.subplots(ncols=2, figsize=(15, 20))

ax[0].axis('off')
ax[0].imshow(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY), cmap='gray')
ax[0].title.set_text('opencv convert')
ax[1].axis('off')
ax[1].imshow(np.mean(img_rgb, axis=2), cmap='gray')
ax[1].title.set_text('np.mean')

plt.show()

### **Резюме**

1. С помощью opencv мы можем читать и сохранять изображения
2. Функции imread и imwrite работают с BGR форматом, важно не забывать делать конвертацию в RGB и обратно
3. Для конвертации фото в GrayScale надо использовать метод cv2.cvtColor c аргументом cv2.COLOR_RGB2GRAY, а не просто усреднять интенсивности каждого канала

## **Поработаем с HSV моделью**

Давайте теперь посмотрим, как можно работать с HSV форматом.

In [ ]:
# для конвертации в HSV используем тот же метод cvtColor
# снова обратите внимание, что есть два варианта: cv2.COLOR_BGR2HSV и cv2.COLOR_RGB2HSV
img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

Вспоминаем какой канал за что отвечает:

- **H-Hue** - оттенок, задается углом от 0 до 359, но в opencv приводится к диапазону **от 0 до 179**, чтобы уложиться в uint8
- **S-Saturation** - насыщенность, задается числом от 0 до 255
- **V-Value** - яркость, зададается числом от 0 до 255

Подробности по конвертации в доке https://docs.opencv.org/3.4/de/d25/imgproc_color_conversions.html

In [ ]:
# проверим, что всё действительно так
print(f'H min:{np.min(img_hsv[:, :, 0])}, max:{np.max(img_hsv[:, :, 0])}')
print(f'S min:{np.min(img_hsv[:, :, 1])}, max:{np.max(img_hsv[:, :, 1])}')
print(f'V min:{np.min(img_hsv[:, :, 2])}, max:{np.max(img_hsv[:, :, 2])}')

In [ ]:
# Посмотрим как отреагирует pyplot на такой формат
plt.imshow(img_hsv)
plt.axis('off')
plt.show()

Реакция так себе :)

pyplot ожидает на входе тензор с RGB, поэтому HSV адекватно отобразить не может.

Давайте посмотрим как в HSV решить задачку удаления зелёного фона.

Из лекции мы помним, что делать это в RGB представлении довольно безрезультатно, так что сфокусируемся именно на решении с HSV.

In [ ]:
# посмотрим на гистограмму распределения H
counts, bins = np.histogram(img_hsv[:, :, 0], bins=30)
plt.figure(figsize=(15, 7))
plt.xlabel('h degree')
plt.ylabel('pixel count')
plt.title('Hue Histogram')
plt.hist(bins[:-1], bins, weights=counts, color='green')
plt.show()

In [ ]:
# создаем маску из нашего изображения
# выбираем только те пиксели, что по h попадают в диапазон (50, 60)
mask = (img_hsv[:, :, 0] > 50) & (img_hsv[:, :, 0] < 60)

In [ ]:
# Посмотрим как выглядит такая маска
plt.imshow(mask, cmap='gray')
plt.axis('off')
plt.title('mask for background')
plt.show()

In [ ]:
# Мы хотим оставить пиксели лица, и выбросить при маскировании пиксели фона
# поэтому маску надо инвертировать - взять от нее отрицание
# сделать это можно двумя способами: mask = cv2.bitwise_not(mask)
# либо так:
mask = ~mask

# и снова смотрим что вышло
plt.imshow(mask, cmap='gray')
plt.axis('off')
plt.title('mask for Hokage')
plt.show()

Контуры маски выглядят супер, только внутри дырки, которые мы хотели бы залить.

Давайте попробуем исправить неидеальности маски с помощью морфологий.

In [ ]:
# прежде чем отдавать маску морфологии, давайте сделаем её трехмерной и приведем к нужному типу - uint8
mask = mask.reshape((h,w,1)).astype('uint8')

In [ ]:
# нам надо залить дырки внутри, а значим будем использовать дилатацию
# чтобы это сделать надо задать структурный элемент, а потом с ним провести нужную операцию
# посмотрим какой результат поулчится с ядрами разного размера:
fig, ax = plt.subplots(ncols=3, figsize=(30, 25))

for i, K in enumerate([3, 9, 30]):
    kernel = np.ones((K,K),np.uint8)
    dilated = cv2.dilate(mask, kernel)
    ax[i].axis('off')
    ax[i].title.set_text(f'K=({K}, {K})')
    ax[i].imshow(dilated, cmap='gray')

plt.show()

In [ ]:
# средняя маска выглядит неплохо, давайте её и возьмем
K = 9
kernel = np.ones((K,K), np.uint8)
dilated_mask = cv2.dilate(mask, kernel)
dilated_mask = dilated_mask.reshape((h,w,1)) # добавили размерность, чтобы битовая операция * сработала

In [ ]:
# а теперь сравним результаты 
fig, ax = plt.subplots(ncols=2, figsize=(20, 25))
ax[0].axis('off')
ax[0].imshow(img_rgb * mask)
ax[0].title.set_text('Hokage with Initial Mask')
ax[1].axis('off')
ax[1].imshow(img_rgb * dilated_mask)
ax[1].title.set_text('Hokage with Dilated Mask')
plt.show()

Выглядит круто внутри объекта, но мы захватили лишнего по контуру, попробуем это поправить с помощью эрозии, то есть проведем операцию **закрытия**: дилатация -> эрозия

In [ ]:
# сделаем две маски для наглядности
# одну с помощью дилатации, вторую с дилатацией->эрозией

kernel = np.ones((K,K), np.uint8)
dilated_mask = cv2.dilate(mask, kernel)
closed_mask = cv2.erode(dilated_mask, kernel)

dilated_mask = dilated_mask.reshape((h,w,1)) # добавили размерность, чтобы битовая операция * сработала
closed_mask = closed_mask.reshape((h,w,1)) # добавили размерность, чтобы битовая операция * сработала

In [ ]:
# а теперь сравним результаты 
fig, ax = plt.subplots(ncols=3, figsize=(30, 25))
ax[0].axis('off')
ax[0].imshow(img_rgb * mask)
ax[0].title.set_text('Hokage with Initial Mask')
ax[1].axis('off')
ax[1].imshow(img_rgb * dilated_mask)
ax[1].title.set_text('Hokage with Dilated Mask')
ax[2].axis('off')
ax[2].imshow(img_rgb * closed_mask)
ax[2].title.set_text('Hokage with Eroded Dilated Mask')
plt.show()

#### маска 3 x 3

In [ ]:
# сделаем две маски для наглядности
# одну с помощью дилатации, вторую с дилатацией->эрозией

kernel = np.ones((3, 3), np.uint8)
dilated_mask = cv2.dilate(mask, kernel)
closed_mask = cv2.erode(dilated_mask, kernel)

dilated_mask = dilated_mask.reshape((h,w,1)) # добавили размерность, чтобы битовая операция * сработала
closed_mask = closed_mask.reshape((h,w,1)) # добавили размерность, чтобы битовая операция * сработала

In [ ]:
# а теперь сравним результаты 
fig, ax = plt.subplots(ncols=3, figsize=(30, 25))
ax[0].axis('off')
ax[0].imshow(img_rgb * mask)
ax[0].title.set_text('Hokage with Initial Mask')
ax[1].axis('off')
ax[1].imshow(img_rgb * dilated_mask)
ax[1].title.set_text('Hokage with Dilated Mask')
ax[2].axis('off')
ax[2].imshow(img_rgb * closed_mask)
ax[2].title.set_text('Hokage with Eroded Dilated Mask')
plt.show()

**Ну какова красота!**

### **Резюме**

1. С помощью HSV удобно находить пиксели определенных цветов
2. С помощью логических операций можно создавать маски для для изображения
3. Дилатация может помочь нам избравиться от дырок в бинарных масках, но увеличит их размер
4. Эрозия, использованная после дилатации с тем же ядром (операция закрытия) решает проблему слишком увеличивающихся размеров объектов

## **Выравниваем гистограммы**

В лекции мы познакомились с разными видами эквилизации гистограммы. 

Давайте проведем эксперименты с ними на какой-нибудь действительно проблемной по контрастности картинке. 

In [ ]:
# считаем картинку, переведем в grayscale, чтобы у нас остался только один канал

brain = cv2.imread('brain.png')
brain = cv2.cvtColor(brain, cv2.COLOR_BGR2GRAY)
print(brain.shape)

In [ ]:
plt.imshow(brain, cmap='gray')
plt.axis('off')
plt.title('image')
plt.show()

In [ ]:
# что у такой картинки с гистограммой интенсивности?

counts, bins = np.histogram(brain, bins=50)
F1 = np.cumsum(counts)
plt.figure(figsize=(10, 5))
plt.xlabel('pixel values')
plt.ylabel('pixel count')
plt.title('Intensity Histogram')
plt.plot(bins[:-1], F1, color='black')
plt.hist(bins[:-1], bins, weights=counts, color='red')
plt.show()

In [ ]:
# проведем выравнивание гистограммы
# дока https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html
equ = cv2.equalizeHist(brain)

In [ ]:
# давайте посмотрим что получилось
fig, ax = plt.subplots(ncols=2, figsize=(20, 10))
ax[0].axis('off')
ax[0].imshow(brain, cmap='gray')
ax[0].title.set_text('image')
ax[1].axis('off')
ax[1].imshow(equ, cmap='gray')
ax[1].title.set_text('HE')
plt.show()

А вы по первой картинке заметили, что внутри черепа текстура неоднородна? Я вот нет. После HE не заметить стало невозможно.

In [ ]:
# посмотрим как изменилась гистограмма и кумулятивная функция после HE

counts, bins = np.histogram(equ, bins=50)
F1 = np.cumsum(counts)
plt.figure(figsize=(10, 5))
plt.xlabel('pixel values')
plt.ylabel('pixel count')
plt.title('Intensity Histogram after HE')
plt.plot(bins[:-1], F1, color='black')
plt.hist(bins[:-1], bins, weights=counts, color='red')
plt.show()

AHE - это промежуточная стадия между HE и CLAHE, поэтому давайте сразу использовать второй, наиболее продвинутый метод, а АНЕ тут опустим.

In [ ]:
# создаем преобразование (у него можно задать параметры, но и дефолтные хороши)
clahe = cv2.createCLAHE()

# применяем созданное преобразование к картинке
equ_clahe = clahe.apply(brain)

In [ ]:
# сравниваем все результаты
fig, ax = plt.subplots(ncols=3, figsize=(30, 10))
ax[0].axis('off')
ax[0].title.set_text('image')
ax[0].imshow(brain, cmap='gray')
ax[1].axis('off')
ax[1].title.set_text('HE')
ax[1].imshow(equ, cmap='gray')
ax[2].axis('off')
ax[2].title.set_text('CLAHE')
ax[2].imshow(equ_clahe, cmap='gray')
plt.show()

Тут мы ещё ярче уивдели структуры, которые раньше не просматривались. Круто.

In [ ]:
# сравниваем все гистограммы
fig, ax = plt.subplots(ncols=3, figsize=(30, 7))
titles = ['Intensity Histogram', 'Intensity Histogram after HE', 'Intensity Histogram after CLAHE']

for i, src in enumerate([brain, equ, equ_clahe]):
    counts, bins = np.histogram(src, bins=50)
    F1 = np.cumsum(counts)
    ax[i].set_xlabel('pixel values')
    ax[i].set_ylabel('pixel count')
    ax[i].title.set_text(titles[i])
    ax[i].plot(bins[:-1], F1, color='black')
    ax[i].hist(bins[:-1], bins, weights=counts, color='red')
plt.show()

### **Резюме**

1. C помощью HE мы можем улучшить контрастность изображений
2. А с CLAHE результат получится ещё лучше, всегда берите его

## **Разбираемся с Resize**

Давайте сначала посмотрим на downscale, а потом на обратную ему операцию - upscale.

In [ ]:
# resize в opencv можно проводить двумя способами:
# либо явно указать желаемый финальный размер - dsize

img_small = cv2.resize(img_rgb, dsize=(100, 150))

fig, ax = plt.subplots(ncols=2, figsize=(20, 10))
ax[0].imshow(img_rgb)
ax[0].axis('off')
ax[0].title.set_text(f'img size: {img.shape}')
ax[1].imshow(img_small)
ax[1].title.set_text(f'img size: {img_small.shape}')
ax[1].axis('off')
plt.show()

При таком подходе очень высока вероятность искажения фото из-за изменения соотношения сторон. 

In [ ]:
# Другой метод - это указать в какое число раз хотим изменить каждую из сторон
# если аргументы fx, fy < 1, то делаем downscale, если >1, то upscale

img_small = cv2.resize(img_rgb, dsize=(0, 0), fx=0.1, fy=0.1)

fig, ax = plt.subplots(ncols=2, figsize=(20, 10))
ax[0].imshow(img_rgb)
ax[0].axis('off')
ax[0].title.set_text(f'{img.shape}')
ax[1].imshow(img_small)
ax[1].title.set_text(f'{img_small.shape}')
ax[1].axis('off')
plt.show()

In [ ]:
# попробуем сами побороться с алиасингом
# для этого применим сверточную операцию для гауссовского размытия
# её дока https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html

blur = cv2.GaussianBlur(img_rgb, (15, 15), 0) 
img_small_anti = cv2.resize(blur, dsize=(0,0), fx=0.1, fy=0.1)

fig, ax = plt.subplots(ncols=3, figsize=(30, 10))
for i, src in enumerate([img_rgb, img_small, img_small_anti]):
    ax[i].imshow(src)
    ax[i].axis('off')
    ax[i].title.set_text(f'{src.shape}')
    
plt.show()

Круто, результат намного лучше и никаких артефактов на границах.

### А можем ли мы решить проблему используя другие методы интерполяции?

Давайте протестируем такие варианты:
- ```cv2.INTER_NEAREST``` — интерполяция методом ближайшего соседа (nearest-neighbor interpolation),
- ```cv2.INTER_LINEAR``` — билинейная интерполяция (bilinear interpolation (используется по умолчанию),
- ```cv2.INTER_CUBIC``` — бикубическая интерполяция (bicubic interpolation) в окрестности 4×4 пикселей,
- ```cv2.INTER_AREA``` — передискретизация с использованием отношения площади пикселя,
- ```cv2.INTER_LANCZOS4``` — интерполяция Ланцоша (Lanczos interpolation) в окрестности 8×8 пикселей.

In [ ]:
fig, ax = plt.subplots(ncols=5, figsize=(30, 10))

for i, method in enumerate([cv2.INTER_NEAREST, cv2.INTER_LINEAR, cv2.INTER_CUBIC, cv2.INTER_AREA, cv2.INTER_LANCZOS4]):
    img_small = cv2.resize(img_rgb, dsize=(0,0), fx=0.1, fy=0.1, interpolation=method)
    ax[i].imshow(img_small)
    ax[i].axis('off')
    ax[i].title.set_text(f'{img_small.shape}')
    
plt.show()

Как мы и видели в примерах на лекции только INTER_AREA метод сам борется с алиасингом, поэтому в общем случае это лучший выбор для downscale. 

Что в методе AREA происходит можно почитать в этом материале:  https://disk.yandex.ru/i/pxl9PLfrdDe8SQ

In [ ]:
# Теперь давайте попробуем вернуть к начальному размеру самый удачный вариант
img_small = cv2.resize(img_rgb, dsize=(0,0), fx=0.1, fy=0.1, interpolation=cv2.INTER_AREA)

fig, ax = plt.subplots(ncols=5, figsize=(30, 10))

for i, method in enumerate([cv2.INTER_NEAREST, cv2.INTER_LINEAR, cv2.INTER_CUBIC, cv2.INTER_AREA, cv2.INTER_LANCZOS4]):
    img_big = cv2.resize(img_small, dsize=(0,0), fx=10, fy=10, interpolation=method)
    ax[i].imshow(img_big)
    ax[i].axis('off')
    ax[i].title.set_text(f'{img_big.shape}')
    
plt.show()

Тут билинейная интерполяция выглядит самой замыленной, зато она довольно быстрая (быстрее только NN).  
По умолчанию именно ```INTER_LINEAR``` используется в opencv для Resize.

Также мы знаем, что уже знаем, что повороты тоже делают с помощью интерполяции. Но не все. 

Если угол поворота кратен 90, то необходимости в интерполяции нет, это будет просто операция транспонирования/обращения порядка и для неё в opencv предусмотрен метод ```cv2.rotate()```

In [ ]:
img_90 = cv2.rotate(img_rgb, cv2.ROTATE_90_CLOCKWISE) # на 90 по часовой
img_180 = cv2.rotate(img_rgb, cv2.ROTATE_180) # на 180
img_270 = cv2.rotate(img_rgb, cv2.ROTATE_90_COUNTERCLOCKWISE) # на 270 по часовой (90 против часовой)

In [ ]:
# Посмотрим что получилось
angles = [0, 90, 180, 270]
fig, ax = plt.subplots(ncols=4, figsize=(30, 10))

for i, vis_img in enumerate([img_rgb, img_90, img_180, img_270]):
    ax[i].imshow(vis_img)
    ax[i].axis('off')
    ax[i].title.set_text(f'Angle: {angles[i]}, Size: {vis_img.shape}')
    
plt.show()

Эти же операции мы можем проделать просто с массивом методами numpy. 

In [ ]:
img_90_np = np.transpose(img_rgb, axes=[1,0,2])[:, ::-1]
img_180_np = img_rgb[::-1, ::-1] # берем пиксели в обратном порядке по вертикали и горизонтали
img_270_np = img_90_np[::-1, ::-1] # берем пиксели в обратном порядке по горизонтали для транспонированного изображения

In [ ]:
# Посмотрим что теперь получилось
angles = [0, 90, 180, 270]
fig, ax = plt.subplots(ncols=4, figsize=(30, 10))

for i, vis_img in enumerate([img_rgb, img_90_np, img_180_np, img_270_np]):
    ax[i].imshow(vis_img)
    ax[i].axis('off')
    ax[i].title.set_text(f'Angle: {angles[i]}, Size: {vis_img.shape}')
    
plt.show()

Видим, что при транспонировании поворот на 90 у нас произошел против часовой стрелки и изображение оказалось отзеркалено. 

Конечно, мы можем это легко поправить и снова взять пиксели в другом порядке, но возиться будет не охота, просто используйте ```cv2.rotate()```

Давайте теперь разбираться что происход при вращении на произвольный угол. 

In [ ]:
h, w, c = img_rgb.shape

# для того, чтобы осуществить вращение, нужно задать матрицу вращения

# для нее мы задаем точку center, вокруг которой будем вращать
# угол angle, который может быть положительным и отрицательным (знак регулирует направление, - это по часовой стрелке)
# масштаб scale, на который будем увеличивать/уменьшать изображение
# инфа и примеры тут https://www.geeksforgeeks.org/python-opencv-getrotationmatrix2d-function/

M = cv2.getRotationMatrix2D(center=(w/2, h/2), angle=-30, scale=1) 

# на основании матрицы будем выполнять аффинное преобразование нашей картинки
# для него мы задаем финальный размер изображения dsize, а ещё можем задать тип интерполяции
rotate_m_30 = cv2.warpAffine(img_rgb, M, dsize=(w, h)) 

In [ ]:
# попробуем ещё варианты
M = cv2.getRotationMatrix2D(center=(w/2, h/2), angle=45, scale=10) 
rotate_45 = cv2.warpAffine(img_rgb, M, dsize=(w, h)) 

M = cv2.getRotationMatrix2D(center=(w/2, h/2), angle=45, scale=0.5) 
rotate_45_sc2 = cv2.warpAffine(img_rgb, M, dsize=(w, h)) 

In [ ]:
# Посмотрим на всё что получилось
angles = [0, -30, 45, 45]
fig, ax = plt.subplots(ncols=4, figsize=(30, 10))

for i, vis_img in enumerate([img_rgb, rotate_m_30, rotate_45, rotate_45_sc2]):
    ax[i].imshow(vis_img)
    ax[i].axis('off')
    ax[i].title.set_text(f'Angle: {angles[i]}, Size: {vis_img.shape}')
    
plt.show()

Таким образом, чтобы повернуть изображение, надо провести над ним аффинное преобразование, в котором мы сразу и скейлинг можем задать. 

Для того, чтобы новое изображение вписалось в границы, можно варьировать параметры размеров в ```cv2.warpAffine``` и скейла в ```cv2.getRotationMatrix2D```.

### **Резюме**

1. C помощью метода cv2.Resize мы можем как угодно ресайзить изображение (увеличивать/уменьшать/менять соотношение сторон) 
2. Для Downsample лучше всего использовать метод интерполяции cv2.INTER_AREA, так как только он по умолчанию не подвержен алиасингу
3. Для Upsample изображения в целом можно использовать любой метод, cv2.INTER_LINEAR будет самым быстрым из них (но не самым точным)
4. Повороты на 90*N градусов можно делать просто с помощью транспонирования тензора и переставления пикселей, либо метода cv2.rotate()
5. Для поворота на произвольный градус надо задавать матрицу поворота и использовать аффинное преобразование, в этом случае надо думать про границы нового изображения

## **Выделяем границы и контуры**

В лекции мы уже видели подробные результаты по оператору Собеля, если хочется воскресить, то вот дока https://docs.opencv.org/4.x/d5/d0f/tutorial_py_gradients.html

А в этом разделе давайте сразу использовать Кэнни, и попробуем с помощью нескольких алгоритмов задетектировать номер на машине.

In [ ]:
img = cv2.imread('car_1.jpg')

In [ ]:
# чтобы найти границы надо перейти в GrayScale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# ищем границы
edges = cv2.Canny(gray, 100, 200) # тут два параметра - t и T для двойной пороговой фильтрации

# попробуем сперва поблюрить изображение, а потом уже найти границы
gray_blurred = cv2.GaussianBlur(gray, (9, 9), 0)
edges_blur = cv2.Canny(gray_blurred, 100, 200)

In [ ]:
# сравниваем результаты
titles = ['img', 'Canny img edges', 'Canny blurred img edges']

fig, ax = plt.subplots(ncols=3, figsize=(30, 10))
for i, src in enumerate([gray, edges, edges_blur]):
    ax[i].imshow(src, cmap='gray')
    ax[i].axis('off')
    ax[i].title.set_text(titles[i])
plt.show()

Видим, что последний вариант для нас намного лучше - тут остались только самые важные границы, а значит у нас больше шансов выделить из них нужный нам номер.

In [ ]:
# воспользуемся методом для поиска контуров
# дока https://docs.opencv.org/3.4/d4/d73/tutorial_py_contours_begin.html

cnts, _ = cv2.findContours(edges_blur, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
# сколько же контуров у нас нашлось?
len(cnts)

In [ ]:
# Теперь мы хотим найти такие контуры, что аппроксимируются черетыхугольником
# дока https://docs.opencv.org/4.x/dc/dcf/tutorial_js_contour_features.html

# отсортируем их по уменьшению площали
cnts = sorted(cnts, key = cv2.contourArea, reverse = True)

result = None
for c in cnts:
    # считаем пермиметр чтобы задать погрешность для аппроксимации
    peri = cv2.arcLength(c, True)
    
    # считаем аппроксимацию полигоном, задаем для нее максимальную погрешность epsilon = 0.02 * peri
    approx = cv2.approxPolyDP(c, epsilon=0.02 * peri, closed=True)
    
    if len(approx) == 4: # если полигон имеет 4 вершины, это наш пациент, прекращаем процедуру
        result = approx
        break

In [ ]:
# посмотрим нашлось ли что-то
result

In [ ]:
# с надеждой смотрим на результат
image_to_draw = img.copy()
cv2.drawContours(image_to_draw, [result], -1, (0, 255, 0), 3)
plt.imshow(cv2.cvtColor(image_to_draw, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

### Та-дааам!

### **Резюме**

1. Метод cv2.Canny очень эффективно находить границы
2. Чтобы найти только самые значимые границы и снизить шум, до Canny можно размыть изображение с помощью cv2.GaussianBlur
3. По найденным границам с помощью cv2.findContours мы можем найти все возможные контуры (обычно их очень много)
4. Проверка на точную аппроксимируемость какой-либо фигурой может нам помочь найти нужные фигуры на фото (например, полигоны)

## **Гомография**

Теперь давайте попробуем повторить наш алгоритм для другого изображения.

In [ ]:
img = cv2.imread('car_2.jpg')

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray_blurred = cv2.GaussianBlur(gray, (7, 7), 0) # тут я немного уменьшила ядро, тк (9, 9) приводило к разрыву контура номера
edges_blur = cv2.Canny(gray_blurred, 100, 200)

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(30, 10))
for i, src in enumerate([gray, edges_blur]):
    ax[i].imshow(src, cmap='gray')
    ax[i].axis('off')
plt.show()

In [ ]:
# находим снова все контуры
cnts, _ = cv2.findContours(edges_blur, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
print(len(cnts))

In [ ]:
# сортируем их по площади и берем первый хорошо аппроксимирующийся прямоугольником
cnts = sorted(cnts, key = cv2.contourArea, reverse = True)

result = None
for c in cnts:
    peri = cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, 0.02 * peri, True)
    if len(approx) == 4: 
        result = approx
        break

In [ ]:
# смотрим что вышло
image_to_draw = img.copy()
cv2.drawContours(image_to_draw, [result], -1, (0, 255, 0), 3)
plt.imshow(cv2.cvtColor(image_to_draw, cv2.COLOR_BGR2RGB))
#plt.axis('off')
plt.show()

Супер, теперь было бы круто выровнять номер (чтобы потом его распознать, например). 

Для этого нам надо найти матрицу гомографии. 

Матрица гомографии задается 4-мя начальными точками (они у нас уже есть) и 4-мя конечными точками (их надо высчитать самим).

In [ ]:
# решейпим точки контура в нужную размерность, это наши начальные точки
src = result.reshape((4, 2))

In [ ]:
# давайте посмотрим, как устроен контур
# тут важен порядок - точки идут по контуру друг за другом против часовой стрелки
print(src)

In [ ]:
# считаем новые координаты для наших 4х точек

# высчитываем две ширины и выбираем максимальную
width_AD = np.sqrt(((src[0][0] - src[3][0]) ** 2) + ((src[0][1] - src[3][1]) ** 2))
width_BC = np.sqrt(((src[1][0] - src[2][0]) ** 2) + ((src[1][1] - src[2][1]) ** 2))
w = max(int(width_AD), int(width_BC))
 
# высчитываем две высоты и выбираем максимальную
height_AB = np.sqrt(((src[0][0] - src[1][0]) ** 2) + ((src[0][1] - src[1][1]) ** 2))
height_CD = np.sqrt(((src[2][0] - src[3][0]) ** 2) + ((src[2][1] - src[3][1]) ** 2))
h = max(int(height_AB), int(height_CD))

In [ ]:
# очень важно тут правильно задать новое положение для каждой точки - так же против часовой стрелки
dst = np.intp([[0, 0], [0, h], [w, h], [w, 0]])

In [ ]:
# находим гомографию
# ещё примеры https://docs.opencv.org/4.x/d1/de0/tutorial_py_feature_homography.html
H, status = cv2.findHomography(src, dst)

# и по матрице гомографии выправляем перспективу
# объяснения тут https://theailearner.com/tag/cv2-warpperspective/
imgCropped = cv2.warpPerspective(img, H, (int(w), int(h)))

In [ ]:
# с надежой смотрим на то, что получилось
plt.imshow(imgCropped)
plt.axis('off')
plt.show()

### Та-дааам!

### **Резюме**

1. C помощью гомографии мы можем довольно просто провести перспективное преобразование плоскости
2. Зная координаты 4х начальных и 4х конечных точек, методом ```cv2.findHomography``` мы можем автоматически посчитать матрицу гомографии
3. А с матрицей гомографии перспетива исправляется просто методом ```cv2.warpPerspective```

## **Итоги**

Итак, мы посмотрели на практике в этом ноутбуке:
- как читать, сохранять и конвертировать изображения
- как маскировать изображения
- как применять морфологии
- как делать выравнивание гистограммы
- как находить граница объектов и контуры
- как аппроксимировать контуры полигонами
- как находить гомографию и исправлять перспективы

Ну какие же мы молодцы!

### 2026.04.26 more examples

In [ ]:
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt

img = cv.imread('gradients.png', cv.IMREAD_GRAYSCALE)
assert img is not None, "file could not be read, check with os.path.exists()"

laplacian = cv.Laplacian(img,cv.CV_64F)
sobelx = cv.Sobel(img,cv.CV_64F,1,0,ksize=5)
sobely = cv.Sobel(img,cv.CV_64F,0,1,ksize=5)

plt.subplot(2,2,1),plt.imshow(img,cmap = 'gray')
plt.title('Original'), plt.xticks([]), plt.yticks([])
plt.subplot(2,2,2),plt.imshow(laplacian,cmap = 'gray')
plt.title('Laplacian'), plt.xticks([]), plt.yticks([])
plt.subplot(2,2,3),plt.imshow(sobelx,cmap = 'gray')
plt.title('Sobel X'), plt.xticks([]), plt.yticks([])
plt.subplot(2,2,4),plt.imshow(sobely,cmap = 'gray')
plt.title('Sobel Y'), plt.xticks([]), plt.yticks([])

plt.show()

In [ ]:
edges = cv2.Canny(img, 10, 60)

plt.subplot(2, 1, 1)
plt.imshow(img, cmap='gray')

plt.subplot(2, 1, 2)
plt.imshow(edges, cmap='gray')

## 2026.04.29

In [ ]:
import cv2
import numpy as np

# 1. Создаём синтетическое изображение с несколькими фигурами
img = np.zeros((400, 600, 3), dtype=np.uint8)
# Рисуем квадрат
cv2.rectangle(img, (50, 50), (150, 150), (255, 255, 255), -1)
# Рисуем круг
cv2.circle(img, (300, 100), 50, (255, 255, 255), -1)
# Рисуем пятиугольник
pts = np.array([[500, 50], [560, 100], [540, 170], [460, 170], [440, 100]], np.int32)
cv2.fillPoly(img, [pts], (255, 255, 255))

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. Находим границы (утверждение 2 – здесь именно поиск границ, а не аппроксимация)
edges = cv2.Canny(gray, 50, 150)

# 3. Находим контуры (это уже границы, но в виде набора точек)
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 4. Для каждого контура пробуем аппроксимировать его разными фигурами (утверждение 1)
for contour in contours:
    # Аппроксимация многоугольником (полигоном) – упрощаем контур до отрезков
    epsilon = 0.02 * cv2.arcLength(contour, True)  # точность аппроксимации
    approx = cv2.approxPolyDP(contour, epsilon, True)

    # ---------- Проверка на аппроксимируемость разными фигурами (утверждение 3) ----------
    # Проверка на прямоугольник (4 вершины)
    if len(approx) == 4:
        # Дополнительная проверка на прямоугольность (углы ~90°)
        # Здесь упрощённо: считаем, что если 4 точки, это квадрат/прямоугольник
        print("Найден прямоугольник/квадрат с аппроксимацией")
        # Можно извлечь координаты и нарисовать
        cv2.drawContours(img, [approx], 0, (0, 255, 0), 2)

    # Проверка на круг (через вписанную окружность)
    elif len(approx) > 8:  # много точек – возможно, круг
        # Вычисляем площадь исходного контура и площадь аппроксимирующего круга
        area = cv2.contourArea(contour)
        (x, y), radius = cv2.minEnclosingCircle(contour)
        circle_area = 3.14159 * radius * radius
        if abs(area - circle_area) / area < 0.2:  # допуск 20%
            print("Найден круг")
            cv2.circle(img, (int(x), int(y)), int(radius), (255, 0, 0), 2)

# Показать результат
cv2.imshow("Original with detections", img)
cv2.waitKey(0)
cv2.destroyAllWindows()